# Análisis VIC - Cuenca Paucartambo, Pasco - Peru

Notebook interactivo para análisis post-simulación del modelo VIC.

**Estructura del proyecto:**
1. Verificar outputs del modelo
2. Balance hídrico
3. Análisis de caudales
4. Mapas espaciales
5. Análisis de sensibilidad

In [ ]:
import sys
sys.path.insert(0, '/app/scripts')

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
import yaml
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

# Rutas
DATA_DIR = Path('/data')
PLOTS_DIR = Path('/app/plots')
PLOTS_DIR.mkdir(exist_ok=True)

print('Librerías cargadas correctamente.')

In [ ]:
# Cargar configuración
with open('/app/config/basin_config.yml') as f:
    config = yaml.safe_load(f)

basin = config['basin']
outlet = basin['outlet']
print(f"Cuenca: {basin['name']}, {basin['department']}, {basin['country']}")
print(f"Outlet: {outlet['name']} ({outlet['lat']:.3f}°S, {outlet['lon']:.3f}°W)")

## 1. Cargar datos del modelo

In [ ]:
# Dominio
ds_domain = xr.open_dataset(DATA_DIR / 'domain/domain.nc')
mask = ds_domain['mask'].values == 1
print(f"Dominio: {ds_domain.dims}")
print(f"Celdas activas: {mask.sum():,}")

# Salidas VIC
flux_files = sorted((DATA_DIR / 'outputs').glob('fluxes*.nc'))
print(f"\nArchivos de salida VIC: {len(flux_files)}")
for f in flux_files[:5]:
    print(f'  {f.name}')

if flux_files:
    ds_vic = xr.open_mfdataset(flux_files, combine='by_coords')
    print(f"\nVariables VIC: {list(ds_vic.data_vars)}")
    print(f"Período: {ds_vic.time.values[0]} → {ds_vic.time.values[-1]}")

In [ ]:
# Caudal enrutado
q_file = DATA_DIR / 'routing/streamflow_yuncan.nc'
if q_file.exists():
    ds_q = xr.open_dataset(q_file)
    streamflow = ds_q['streamflow'].to_series()
    print(f"Caudal en Yuncan: {len(streamflow)} días")
    print(f"Media: {streamflow.mean():.1f} m³/s")
    print(f"Máximo: {streamflow.max():.1f} m³/s")
else:
    print("No se encontró archivo de caudal. Ejecutar primero 06_run_routing.py")

## 2. Mapa del dominio

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 8),
                        subplot_kw={'projection': ccrs.PlateCarree()})

bbox = basin['bbox']
ax.set_extent([bbox['lon_min'], bbox['lon_max'],
               bbox['lat_min'], bbox['lat_max']])

# Máscara de cuenca
lats = ds_domain.lat.values
lons = ds_domain.lon.values
mask_plot = ds_domain['mask'].values.astype(float)
mask_plot[mask_plot == 0] = np.nan

ax.pcolormesh(lons, lats, mask_plot,
              cmap='Blues', transform=ccrs.PlateCarree(),
              vmin=0, vmax=2, alpha=0.5)

# Elevación
if 'elev' in ds_domain:
    elev_masked = ds_domain['elev'].where(ds_domain['mask'] == 1)
    im = ax.pcolormesh(lons, lats, elev_masked.values,
                       cmap='terrain', transform=ccrs.PlateCarree(),
                       alpha=0.7, vmin=200, vmax=5000)
    plt.colorbar(im, ax=ax, shrink=0.6, label='Elevación (m)')

# Features
ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle='--')
ax.add_feature(cfeature.RIVERS, linewidth=0.5, edgecolor='blue', alpha=0.6)
ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)

# Outlet
ax.plot(outlet['lon'], outlet['lat'],
        'v', color='red', markersize=12,
        transform=ccrs.PlateCarree(),
        label=outlet['name'], zorder=10)

ax.legend(fontsize=10)
ax.set_title(f"Cuenca {basin['name']} - {basin['department']}, Peru\n"
             f"Resolución: {basin['resolution']}° (~1km)", fontsize=12)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'basin_domain_map.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Balance hídrico

In [ ]:
if 'ds_vic' in dir():
    # Medias espaciales anuales
    vars_to_check = ['OUT_PREC', 'OUT_EVAP', 'OUT_RUNOFF', 'OUT_BASEFLOW']
    available = [v for v in vars_to_check if v in ds_vic.data_vars]
    
    fig, ax = plt.subplots(figsize=(14, 5))
    
    colors = {'OUT_PREC': '#1565C0', 'OUT_EVAP': '#2E7D32',
              'OUT_RUNOFF': '#FF6F00', 'OUT_BASEFLOW': '#6A1B9A'}
    labels = {'OUT_PREC': 'Precipitación', 'OUT_EVAP': 'ET',
              'OUT_RUNOFF': 'Escorrentía', 'OUT_BASEFLOW': 'Flujo base'}
    
    for var in available:
        series = ds_vic[var].mean(dim=['lat', 'lon']).to_series()
        series_monthly = series.resample('ME').sum()
        ax.plot(series_monthly.index, series_monthly.values,
                label=labels[var], color=colors[var], linewidth=1.5)
    
    ax.set_xlabel('Fecha')
    ax.set_ylabel('mm/mes')
    ax.set_title('Balance Hídrico Mensual - Cuenca Paucartambo')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Tabla resumen anual
    print('\nBalance hídrico anual promedio (mm/año):')
    for var in available:
        annual_mean = ds_vic[var].mean(dim=['lat', 'lon']).values.sum() / len(ds_vic.time) * 365
        print(f'  {labels[var]:20s}: {annual_mean:.0f} mm/año')

## 4. Serie temporal de caudal en Yuncan

In [ ]:
if 'streamflow' in dir():
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # Panel 1: Serie completa
    ax1 = axes[0]
    ax1.fill_between(streamflow.index, streamflow.values, alpha=0.3, color='steelblue')
    ax1.plot(streamflow.index, streamflow.values, color='steelblue', linewidth=0.5)
    q_smooth = streamflow.rolling(30).mean()
    ax1.plot(q_smooth.index, q_smooth.values, 'r-', linewidth=2, label='Media 30d')
    ax1.set_ylabel('Caudal (m³/s)')
    ax1.set_title(f'Caudal en {outlet["name"]}')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Panel 2: Ciclo estacional
    ax2 = axes[1]
    seasonal = streamflow.groupby(streamflow.index.month)
    q_mean = seasonal.mean()
    q_std = seasonal.std()
    months = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun',
              'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
    
    ax2.fill_between(range(1, 13),
                     q_mean - q_std, q_mean + q_std,
                     alpha=0.2, color='steelblue')
    ax2.plot(range(1, 13), q_mean.values, 'o-',
             color='steelblue', linewidth=2, markersize=8)
    ax2.set_xticks(range(1, 13))
    ax2.set_xticklabels(months)
    ax2.set_ylabel('Caudal (m³/s)')
    ax2.set_title('Régimen Estacional')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'caudal_yuncan_notebook.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'\nEstadísticas de caudal en {outlet["name"]}:')
    print(f'  Media:    {streamflow.mean():.1f} m³/s')
    print(f'  Mediana:  {streamflow.median():.1f} m³/s')
    print(f'  Máximo:   {streamflow.max():.0f} m³/s ({streamflow.idxmax().date()})')
    print(f'  Q10:      {streamflow.quantile(0.90):.1f} m³/s')
    print(f'  Q50:      {streamflow.quantile(0.50):.1f} m³/s')
    print(f'  Q90:      {streamflow.quantile(0.10):.1f} m³/s')

## 5. Mapa espacial de una variable

In [ ]:
if 'ds_vic' in dir():
    # Media anual de escorrentía
    var = 'OUT_RUNOFF'
    if var in ds_vic.data_vars:
        annual_mean = ds_vic[var].mean(dim='time').where(ds_domain['mask'] == 1)
        
        fig, ax = plt.subplots(1, 1, figsize=(10, 8),
                               subplot_kw={'projection': ccrs.PlateCarree()})
        
        ax.set_extent([bbox['lon_min'], bbox['lon_max'],
                       bbox['lat_min'], bbox['lat_max']])
        
        im = ax.pcolormesh(
            lons, lats, annual_mean.values,
            cmap=cmocean.cm.rain,
            vmin=0, vmax=np.nanpercentile(annual_mean.values, 95),
            transform=ccrs.PlateCarree()
        )
        
        plt.colorbar(im, ax=ax, shrink=0.6, label='Escorrentía media diaria (mm/day)')
        
        ax.add_feature(cfeature.RIVERS, linewidth=0.5, edgecolor='blue', alpha=0.5)
        ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
        ax.plot(outlet['lon'], outlet['lat'], 'v', color='red',
                markersize=12, transform=ccrs.PlateCarree(),
                label=outlet['name'])
        ax.legend(fontsize=9)
        ax.set_title('Escorrentía Media Anual - Cuenca Paucartambo', fontsize=12)
        
        plt.tight_layout()
        plt.show()

## 6. Tiempos de viaje a Yuncan (Opcional)

In [ ]:
import json
tt_file = DATA_DIR / 'routing/travel_times_to_yuncan.json'
if tt_file.exists():
    with open(tt_file) as f:
        travel_times = json.load(f)
    
    print('Tiempos de viaje al embalse Yuncan:')
    for item in travel_times:
        print(f"  {item['reservoir']:35s}: {item['travel_time_days']:.1f} días "
              f"({item['travel_time_hours']:.0f} horas)")
else:
    print('Ejecutar: python scripts/06_run_routing.py --travel-time')